<img src="https://res.cloudinary.com/dtizipxds/image/upload/q_auto/f_auto/v1776264397/banner_yvwajv.png" width="100%">


In [1]:
%pip install numpy pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


# Microsoft Malware - Preprocessing Exercise

This notebook preprocesses malware features, trains a baseline classifier, and evaluates predictions on `test.csv` against `solution.csv`.


## 1) Imports and setup

Import all required libraries and configure notebook display/warning behavior.


In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

## 2) Load exercise files

Read `train.csv`, `test.csv`, and `solution.csv`, and validate file presence.


In [3]:
TRAIN_PATH = Path("train.csv")
TEST_PATH = Path("test.csv")
SOLUTION_PATH = Path("solution.csv")

for p in [TRAIN_PATH, TEST_PATH, SOLUTION_PATH]:
    assert p.exists(), f"Missing file: {p.resolve()}"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
solution_df = pd.read_csv(SOLUTION_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Solution shape: {solution_df.shape}")
train_df.head()


Train shape: (8694, 69)
Test shape: (2174, 68)
Solution shape: (2174, 69)


,asm_commands_add,asm_commands_call,asm_commands_cdq,asm_commands_cld,asm_commands_cli,asm_commands_cmc,asm_commands_cmp,asm_commands_cwd,asm_commands_daa,asm_commands_dd,asm_commands_dec,asm_commands_dw,asm_commands_endp,asm_commands_faddp,asm_commands_fchs,asm_commands_fdiv,asm_commands_fdivr,asm_commands_fistp,asm_commands_fld,asm_commands_fstp,asm_commands_fword,asm_commands_fxch,asm_commands_imul,asm_commands_in,asm_commands_inc,asm_commands_ins,asm_commands_jb,asm_commands_je,asm_commands_jg,asm_commands_jl,asm_commands_jmp,asm_commands_jnb,asm_commands_jno,asm_commands_jo,asm_commands_jz,asm_commands_lea,asm_commands_mov,asm_commands_mul,asm_commands_not,asm_commands_or,asm_commands_out,asm_commands_outs,asm_commands_pop,asm_commands_push,asm_commands_rcl,asm_commands_rcr,asm_commands_rep,asm_commands_ret,asm_commands_rol,asm_commands_ror,asm_commands_sal,asm_commands_sar,asm_commands_sbb,asm_commands_scas,asm_commands_shl,asm_commands_shr,asm_commands_sidt,asm_commands_stc,asm_commands_std,asm_commands_sti,asm_commands_stos,asm_commands_sub,asm_commands_test,asm_commands_wait,asm_commands_xchg,asm_commands_xor,line_count_asm,size_asm,Class
0,455,241.0,0.0,1.0,1.0,0.0,187.0,0.0,18.0,3334,39.0,1812.0,48.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,1.0,0.0,4.0,188,5.0,141.0,9.0,7.0,8.0,13.0,73.0,17.0,0.0,5.0,78.0,93.0,2574.0,5.0,4.0,2177,3.0,1.0,34.0,394.0,0.0,1.0,0.0,34.0,7.0,5.0,0.0,20.0,7.0,2.0,31.0,1.0,0.0,1.0,64.0,3.0,2.0,838.0,6.0,0.0,8.0,14.0,7937,460288,8
1,452,214.0,0.0,2.0,1.0,2.0,183.0,1.0,11.0,3346,30.0,1794.0,50.0,0.0,0.0,3.0,0.0,0.0,1.0,1.0,1.0,0.0,2.0,180,6.0,140.0,3.0,6.0,12.0,12.0,70.0,16.0,0.0,2.0,77.0,95.0,2569.0,5.0,4.0,2157,3.0,1.0,30.0,389.0,1.0,1.0,1.0,32.0,6.0,5.0,0.0,19.0,1.0,1.0,30.0,2.0,0.0,0.0,65.0,3.0,1.0,822.0,6.0,0.0,7.0,12.0,7937,460288,8
2,1139,701.0,33.0,18.0,5.0,0.0,626.0,3.0,671.0,16723,1023.0,1437.0,139.0,42.0,3.0,10.0,0.0,0.0,219.0,138.0,0.0,44.0,33.0,1425,196.0,5.0,145.0,22.0,47.0,75.0,390.0,59.0,0.0,0.0,437.0,493.0,2981.0,122.0,52.0,3996,28.0,0.0,710.0,1316.0,3.0,3.0,46.0,445.0,28.0,62.0,18.0,44.0,14.0,0.0,98.0,47.0,0.0,18.0,213.0,0.0,35.0,1489.0,542.0,7.0,0.0,529.0,76615,4443612,2
3,370,136.0,10.0,9.0,18.0,12.0,143.0,16.0,193.0,47273,360.0,4654.0,9.0,1.0,0.0,5.0,1.0,0.0,8.0,9.0,40.0,2.0,41.0,161,125.0,4.0,10.0,3.0,6.0,2.0,111.0,5.0,5.0,3.0,1.0,70.0,600.0,15.0,23.0,689,1.0,1.0,202.0,358.0,5.0,12.0,5.0,754.0,13.0,12.0,13.0,11.0,94.0,28.0,7.0,11.0,0.0,9.0,114.0,14.0,21.0,227.0,65.0,8.0,133.0,121.0,35073,2034176,6
4,112,199.0,0.0,0.0,0.0,0.0,4.0,0.0,1.0,180,18.0,174.0,41.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,84,0.0,2.0,3.0,5.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,33.0,89.0,0.0,4.0,264,0.0,0.0,134.0,212.0,0.0,0.0,0.0,26.0,0.0,4.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,34.0,0.0,0.0,323.0,3.0,4.0,1.0,51.0,146689,8507904,3


## 3) Quick data quality check

Review data types, missing values, and duplicate rows in the training data.


In [4]:
print("Data types summary:")
print(train_df.dtypes.value_counts())

print("\nMissing values (top 20):")
missing = train_df.isna().sum().sort_values(ascending=False)
print(missing.head(20))

print("\nDuplicate rows:", train_df.duplicated().sum())


Data types summary:
float64    62
int64       7
Name: count, dtype: int64

Missing values (top 20):
asm_commands_add     0
asm_commands_rcl     0
asm_commands_sal     0
asm_commands_ror     0
asm_commands_rol     0
asm_commands_ret     0
asm_commands_rep     0
asm_commands_rcr     0
asm_commands_push    0
asm_commands_lea     0
asm_commands_pop     0
asm_commands_outs    0
asm_commands_out     0
asm_commands_or      0
asm_commands_not     0
asm_commands_mul     0
asm_commands_sar     0
asm_commands_sbb     0
asm_commands_scas    0
asm_commands_shl     0
dtype: int64

Duplicate rows: 58


## 4) Define target summary

Set `Class` as target and inspect class distribution in the training set.


In [5]:
# TODO: set the target column name
# TARGET_COL = "Class"
TARGET_COL = ...
assert TARGET_COL in train_df.columns, f"Target column '{TARGET_COL}' not found in train.csv"
assert TARGET_COL in solution_df.columns, f"Target column '{TARGET_COL}' not found in solution.csv"
assert list(test_df.columns) == [c for c in solution_df.columns if c != TARGET_COL], "test.csv must match solution.csv without target"

class_counts = train_df[TARGET_COL].value_counts().sort_index()
class_ratio = (class_counts / len(train_df)).round(4)

summary = pd.DataFrame({
    "count": class_counts,
    "ratio": class_ratio,
})
summary


AssertionError: Target column 'Ellipsis' not found in train.csv

## 5) Prepare feature matrix and target vector

Split inputs (`X`) and labels (`y`) and confirm feature count.


In [ ]:
X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL].astype("int64")

numeric_cols = X.columns.tolist()
print(f"Numeric feature count: {len(numeric_cols)}")


Numeric feature count: 68


## 6) Build numeric preprocessing pipeline

Apply median imputation, `log1p` transformation, and scaling; then transform fit/validation/test sets.


In [ ]:
# Many count-like features are heavy-tailed; log1p usually stabilizes scale.
def safe_log1p(arr):
    arr = np.asarray(arr)
    return np.log1p(np.clip(arr, a_min=0, a_max=None))

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(safe_log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
    ],
    remainder="drop",
)

# Internal validation split from train.csv
# TODO: split into fit/validation sets using train_test_split
# X_fit, X_val, y_fit, y_val = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )
X_fit, X_val, y_fit, y_val = ...

X_fit_prep = preprocessor.fit_transform(X_fit)
X_val_prep = preprocessor.transform(X_val)
# TODO: transform test features using the fitted preprocessor
# X_test_prep = preprocessor.transform(test_df)
X_test_prep = ...

print("Fit processed shape:", X_fit_prep.shape)
print("Validation processed shape:", X_val_prep.shape)
print("Test processed shape:", X_test_prep.shape)


Fit processed shape: (6955, 68)
Validation processed shape: (1739, 68)
Test processed shape: (2174, 68)


## 7) Train baseline model

Fit a random forest classifier and evaluate performance on validation data.


In [ ]:
# Baseline model to verify preprocessing and data handling.
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
)
model.fit(X_fit_prep, y_fit)

val_pred = model.predict(X_val_prep)
print("Validation Weighted F1:", round(f1_score(y_val, val_pred, average="weighted"), 4))
print(classification_report(y_val, val_pred))


Validation Weighted F1: 0.9902
              precision    recall  f1-score   support

           1       0.95      1.00      0.97       247
           2       1.00      0.99      1.00       396
           3       1.00      1.00      1.00       471
           4       0.97      1.00      0.99        76
           5       1.00      1.00      1.00         7
           6       0.99      0.97      0.98       120
           7       1.00      1.00      1.00        64
           8       1.00      0.94      0.97       196
           9       0.99      1.00      1.00       162

    accuracy                           0.99      1739
   macro avg       0.99      0.99      0.99      1739
weighted avg       0.99      0.99      0.99      1739



## 8) Package outputs and final exercise check

Save processed artifacts, predict on `test.csv`, and compare with `solution.csv`.


In [ ]:
# Optional: package transformed datasets for downstream modeling.
processed = {
    "X_fit": X_fit_prep,
    "X_val": X_val_prep,
    "X_test": X_test_prep,
    "y_fit": y_fit.to_numpy(),
    "y_val": y_val.to_numpy(),
    "feature_names": preprocessor.get_feature_names_out().tolist(),
}

print("Prepared keys:", list(processed.keys()))
print("Feature name sample:", processed["feature_names"][:10])

# Final exercise step: predict test.csv and compare with solution.csv
y_solution = solution_df[TARGET_COL].astype("int64")
# TODO: predict labels for the test set
# test_pred = model.predict(X_test_prep)
test_pred = ...

print("\nTest vs Solution")
print("Test Weighted F1:", round(f1_score(y_solution, test_pred, average="weighted"), 4))
print(classification_report(y_solution, test_pred))


Prepared keys: ['X_fit', 'X_val', 'X_test', 'y_fit', 'y_val', 'feature_names']
Feature name sample: ['num__asm_commands_add', 'num__asm_commands_call', 'num__asm_commands_cdq', 'num__asm_commands_cld', 'num__asm_commands_cli', 'num__asm_commands_cmc', 'num__asm_commands_cmp', 'num__asm_commands_cwd', 'num__asm_commands_daa', 'num__asm_commands_dd']

Test vs Solution
Test Weighted F1: 0.9954
              precision    recall  f1-score   support

           1       0.98      1.00      0.99       308
           2       1.00      1.00      1.00       496
           3       1.00      1.00      1.00       588
           4       1.00      1.00      1.00        95
           5       1.00      0.88      0.93         8
           6       0.99      0.99      0.99       150
           7       1.00      1.00      1.00        80
           8       1.00      0.98      0.99       246
           9       1.00      1.00      1.00       203

    accuracy                           1.00      2174
   macro a